In [6]:
# Install pyspark and java if not already done
!java -version
!pip install pyspark
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless

import os
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col, year, month, when

# Initialize Spark with a config to help debugging
spark = SparkSession.builder \
    .appName("IngestFlightData") \
    .config("spark.sql.debug.maxToStringFields", 1000) \
    .getOrCreate()

print("Spark Session Created Successfully")

openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-124.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-124.04, mixed mode, sharing)


Error: We are still setting things up for you, please try again after the progress bar at the top of the Studio disappears.
Error: We are still setting things up for you, please try again after the progress bar at the top of the Studio disappears.


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/11 14:54:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session Created Successfully


# 1. Define Explicit Schema

In [7]:
# The raw CSV headers (e.g., 'FlightDate') don't match the schema we want ('FL_DATE').
# Also, there are 100+ columns, so I'm creating a mapping dictionary to 
# rename and cast them to the correct types in one go.

column_mapping = {
    "FL_DATE": ("FlightDate", DateType()),
    "OP_CARRIER": ("Reporting_Airline", StringType()),
    "OP_CARRIER_FL_NUM": ("Flight_Number_Reporting_Airline", IntegerType()),
    "ORIGIN_CITY": ("OriginCityName", StringType()),
    "ORIGIN": ("Origin", StringType()),
    "DEST_CITY": ("DestCityName", StringType()),
    "DEST": ("Dest", StringType()),
    "CRS_DEP_TIME": ("CRSDepTime", IntegerType()),
    "DEP_TIME": ("DepTime", FloatType()),
    "DEP_DELAY": ("DepDelay", FloatType()),
    "TAXI_OUT": ("TaxiOut", FloatType()),
    "WHEELS_OFF": ("WheelsOff", FloatType()),
    "WHEELS_ON": ("WheelsOn", FloatType()),
    "TAXI_IN": ("TaxiIn", FloatType()),
    "CRS_ARR_TIME": ("CRSArrTime", IntegerType()),
    "ARR_TIME": ("ArrTime", FloatType()),
    "ARR_DELAY": ("ArrDelay", FloatType()),
    "CANCELLED": ("Cancelled", FloatType()),
    "CANCELLATION_CODE": ("CancellationCode", StringType()),
    "DIVERTED": ("Diverted", FloatType()),
    "CRS_ELAPSED_TIME": ("CRSElapsedTime", FloatType()),
    "ACTUAL_ELAPSED_TIME": ("ActualElapsedTime", FloatType()),
    "AIR_TIME": ("AirTime", FloatType()),
    "DISTANCE": ("Distance", FloatType()),
    "CARRIER_DELAY": ("CarrierDelay", FloatType()),
    "WEATHER_DELAY": ("WeatherDelay", FloatType()),
    "NAS_DELAY": ("NASDelay", FloatType()),
    "SECURITY_DELAY": ("SecurityDelay", FloatType()),
    "LATE_AIRCRAFT_DELAY": ("LateAircraftDelay", FloatType())
}

# Define Path to pick up all 2017 and 2018 files
raw_path = "/teamspace/studios/this_studio/BigDataProject/csv_flight/report_20*.csv"

print("Reading CSVs as raw strings...")
# Read header=True so Spark knows 'FlightDate' exists
df_raw_strings = spark.read.option("header", "true").csv(raw_path)

# Selecting and casting columns based on the dictionary abov
selected_cols = []
for target_name, (source_name, dtype) in column_mapping.items():
    # Logic: Take raw col 'FlightDate', cast to Date, rename to 'FL_DATE'
    selected_cols.append(col(source_name).cast(dtype).alias(target_name))

# Create the Standardized DataFrame
df_raw = df_raw_strings.select(*selected_cols)

# Verify
print(f"Schema applied successfully. Total columns: {len(df_raw.columns)}")
df_raw.printSchema()

Reading CSVs as raw strings...


25/12/11 14:54:24 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /teamspace/studios/this_studio/BigDataProject/csv_flight/report_20*.csv.
java.io.FileNotFoundException: File /teamspace/studios/this_studio/BigDataProject/csv_flight/report_20*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$

Schema applied successfully. Total columns: 29
root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN_CITY: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST_CITY: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: float (nullable = true)
 |-- DEP_DELAY: float (nullable = true)
 |-- TAXI_OUT: float (nullable = true)
 |-- WHEELS_OFF: float (nullable = true)
 |-- WHEELS_ON: float (nullable = true)
 |-- TAXI_IN: float (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: float (nullable = true)
 |-- ARR_DELAY: float (nullable = true)
 |-- CANCELLED: float (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: float (nullable = true)
 |-- CRS_ELAPSED_TIME: float (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: float (nullable = true)
 |-- AIR_TIME: float (nullable

# 2.RDD Transformation

In [8]:
# Requirement: Demonstrate RDD usage.
# I'm converting to RDD to perform a quick quality check (counting cancellations).
# This logic handles potential nulls safely.
raw_rdd = df_raw.rdd

def check_cancelled(row):
    try:
        return 1 if row['CANCELLED'] == 1.0 else 0
    except:
        return 0

# Map/Reduce to count total cancelled flights
cancelled_count = raw_rdd.map(check_cancelled).reduce(lambda x, y: x + y)

print(f"Total Cancelled Flights (Calculated via RDD): {cancelled_count}")

# Filtering out any rows where the Date is completely missing
clean_rdd = raw_rdd.filter(lambda row: row['FL_DATE'] is not None)

# df_stage_1 is our valid dataset moving forward
df_stage_1 = df_raw


Total Cancelled Flights (Calculated via RDD): 199277


# 3.Data Cleaning & Feature Creation

In [9]:
from pyspark.sql.functions import col, year, month, when
# Handling Nulls in Delay Columns (Null implies 0 delay)
delay_cols = ["ARR_DELAY", "DEP_DELAY", "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"]
df_cleaned = df_stage_1.na.fill(0, subset=delay_cols)

# Add Year/Month for Partitioning (feature engineering)
df_cleaned = df_cleaned.withColumn("Year", year(col("FL_DATE"))) \
                       .withColumn("Month", month(col("FL_DATE")))

# Creating a binary target for classification (1 if delay > 15 mins)
df_cleaned = df_cleaned.withColumn("IsDelayed", when(col("ARR_DELAY") > 15, 1).otherwise(0))

# Final Filter
# We only want actual flights, so dropping cancelled ones now
df_final_base = df_cleaned.filter(col("CANCELLED") == 0).drop("CANCELLATION_CODE")

df_final = df_final_base
df_final.printSchema()

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN_CITY: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST_CITY: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: float (nullable = true)
 |-- DEP_DELAY: float (nullable = false)
 |-- TAXI_OUT: float (nullable = true)
 |-- WHEELS_OFF: float (nullable = true)
 |-- WHEELS_ON: float (nullable = true)
 |-- TAXI_IN: float (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: float (nullable = true)
 |-- ARR_DELAY: float (nullable = false)
 |-- CANCELLED: float (nullable = true)
 |-- DIVERTED: float (nullable = true)
 |-- CRS_ELAPSED_TIME: float (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: float (nullable = true)
 |-- AIR_TIME: float (nullable = true)
 |-- DISTANCE: float (nullable = true)
 |-- CARRIER_DELAY: float (nullable = false)
 

# 4.Save Data

In [10]:
# Saving to Parquet for efficient reading in the next notebooks.
# Partitioning by Year/Month allows for faster querying (Partition Pruning).
output_path = "/teamspace/studios/this_studio/BigDataProject/cleaned_flights_parquet"

# Write to Parquet, Partitioned by Year and Month
# This creates a folder structure like: Year=2017/Month=1/part-0000.parquet
df_final.write.mode("overwrite").partitionBy("Year", "Month").parquet(output_path)

print(f"Data successfully saved to {output_path}")

Data successfully saved to /teamspace/studios/this_studio/BigDataProject/cleaned_flights_parquet
